# Loading ERA5 Data

> *Pre-requisites*: Code requires most of the packages listed [here](https://github.com/google-research/arco-era5/tree/main/docs/environment.yml):

We will open the Zarr data with XArray after getting GCS permissions. We can test bucket access with fsspec:

In [1]:
import fsspec

fs = fsspec.filesystem('gs')
fs.ls('gs://weatherbench2/datasets/era5/')

['weatherbench2/datasets/era5/',
 'weatherbench2/datasets/era5/1959-2022-1h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-512x256_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x33.zarr',
 'weatherbench2/datasets/era5/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2',
 'weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg-chu

Next, we'll load the 6-hour downsampled data set

In [41]:
import xarray as xr

reanalysis = xr.open_zarr(
    'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr', 
    chunks={'time': 48},
    consolidated=True,
    decode_timedelta=True    
)

Let's reduce the size by only including data since 1/1/2014 and only our target variables:
- 10m_wind_speed
- 2m_temperature
- snow_depth
- total_precipitation_12hr
- total_precipitation_24hr
- total_precipitation_6hr
- wind_speed

In [42]:
#Select time values since 1/1/2014 and the following variables: 
features = [
    'latitude', 
    'longitude', 
    'time', 
    '10m_wind_speed', 
    '2m_temperature', 
    #'high_vegetation_cover', 
    #'low_vegetation_cover', 
    #'lake_cover', 
    #'leaf_area_index_high_vegetation', 
    #'leaf_area_index_low_vegetation', 
    'snow_depth', 
    'total_precipitation_12hr', 
    'total_precipitation_24hr', 
    'total_precipitation_6hr', 
    #'type_of_high_vegetation', 
    #'type_of_low_vegetation'
]

#reanalysis = reanalysis.sel(time=slice('2014', '2021'))[features]
#Slice reanalysis for just January 2021
reanalysis = reanalysis.sel(time=slice('2021-01-01', '2021-01-02'))[features]



We can convert to a GeoDataFrame and specify a coordinate system, as suggested here: https://gis.stackexchange.com/questions/379877/convert-era5-data-to-wgs84

In [46]:
import geopandas as gpd

reanalysis_counties_df = reanalysis_counties.to_dataframe().reset_index()
gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df['longitude'],
                                                         df['latitude']))

KeyboardInterrupt: 

In [26]:
reanalysis

<xarray.Dataset> Size: 291GB
Dimensions:                   (latitude: 721, longitude: 1440, time: 11688)
Coordinates:
  * latitude                  (latitude) float32 3kB 90.0 89.75 ... -89.75 -90.0
  * longitude                 (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
  * time                      (time) datetime64[ns] 94kB 2014-01-01 ... 2021-...
    point                     (latitude, longitude) float32 4MB 90.0 ... 269.8
Data variables:
    10m_wind_speed            (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>
    2m_temperature            (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>
    snow_depth                (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>
    total_precipitation_12hr  (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>
    total_precipitation_24hr  (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>
    total_precipitation_6hr   (time, latitude, longitude) float32 49GB dask.array<chunksize=(44, 721, 1440), meta=np.ndarray>

Next, we'll create a new coordinate position that combines longitude and latitude into a single coordinate

Next, we'll restrict our data to county centroids (in the counties_centroids.csv file created by the convert_NWS_shapefile_to_county_centroids notebook).

In the ERA5 coordinate system, latitude values are "normal" but longitude values are expressed as values within [0, 360] with respect to the Greenwich Prime Meridian (i.e., instead of [-180, 180]). Since our county centroid data are all West of the Prime Meridian, we can simply adjust their longitude values with an auxiliary function.

In [43]:
#Use xarray to load the file ../Data/counties_centroids.csv
import pandas as pd
counties_centroids = pd.read_csv('../Data/counties_centroids.csv')

counties_centroids['LON'] = counties_centroids['LON'].astype(float)
counties_centroids['LAT'] = counties_centroids['LAT'].astype(float)

#The function below converts "standard" longitude values to ERA5 longitude values
def lon_to_360(dlon: float) -> float:
  return ((360 + (dlon % 360)) % 360)

counties_centroids['LON'] = counties_centroids['LON'].apply(lon_to_360)

In [14]:
#Create a new variable 'position' in counties_centroids that is the ordered pair from LON and LAT
counties_centroids['position'] = list(zip(counties_centroids['LON'], counties_centroids['LAT']))

In [44]:
#Create a new DataArray from counties_centroids with the LON and LAT values as longitude and latitude coordinates and FIPS as the data
counties_centroids_da = xr.DataArray(
    counties_centroids['FIPS'].values,
    coords={
        'longitude': ('points', counties_centroids['LON'].values),
        'latitude': ('points', counties_centroids['LAT'].values)
    },
    dims='points'
)

In [45]:
#Restrict the reanalysis data set to coordinates in counties_centroids_da
reanalysis_counties = reanalysis.sel(
    longitude=reanalysis.longitude.isin(counties_centroids_da.longitude),
    latitude=reanalysis.latitude.isin(counties_centroids_da.latitude)
)

In [7]:
#Convert reanalysis_counties into a pandas dataframe
reanalysis_counties_df = reanalysis_counties.to_dataframe().reset_index()

KeyboardInterrupt: 

We can try saving the zarr file locally as a NetCDF file. The size of the data is roughly 9 GB. However, I've let it run for several hours without it completing.

I've run into roadblocks trying to save locally as a zarr file; I keep getting a "TypeError(f"Expected a BytesBytesCodec. Got {type(data)} instead.")" error. From what I can tell, this appears to be related to zarr 3.

In [14]:
# Compute the size of the reanalysis_counties dataset in GB
#print(f'size: {reanalysis_counties.nbytes / (1024 ** 3)} GiB')

#Export reanalysis_counties to NetCDF
#reanalysis_counties.to_netcdf('../Data/reanalysis_counties.nc')

In [7]:
reanalysis_counties

<xarray.Dataset> Size: 6GB
Dimensions:                   (latitude: 94, longitude: 224, time: 11688)
Coordinates:
  * latitude                  (latitude) float32 376B 48.75 48.5 ... 25.25 24.75
  * longitude                 (longitude) float32 896B 235.8 236.0 ... 292.2
  * time                      (time) datetime64[ns] 94kB 2014-01-01 ... 2021-...
Data variables:
    10m_wind_speed            (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    2m_temperature            (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    snow_depth                (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_12hr  (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_24hr  (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_6hr   (time, latitude, longitude) float32 984MB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>

In [12]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet data set
import pyarrow.parquet as pq
eaglei_outages = pq.read_table('../Data/eaglei_data/eaglei_outages_with_county_info.parquet').to_pandas()

In [13]:
eaglei_outages.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37207946 entries, 0 to 37207945
Data columns (total 33 columns):
 #   Column                            Dtype         
---  ------                            -----         
 0   fips_code                         float64       
 1   customers_out                     float64       
 2   datetime                          datetime64[ns]
 3   YEAR                              int32         
 4   STATE                             object        
 5   CWA                               object        
 6   COUNTYNAME                        object        
 7   FIPS                              float64       
 8   FE_AREA                           object        
 9   centroid                          object        
 10  Pct_Buried_Lines                  float64       
 11  Subregion                         object        
 12  POPULATION                        int64         
 13  BUILDVALUE                        float64       
 14  AGRIVALUE       

In [15]:
#In eaglei_outages convert centroid to the coordinate pair LON, LAT
eaglei_outages['LON'] = eaglei_outages['centroid'].apply(lambda x: x[0])
eaglei_outages['LAT'] = eaglei_outages['centroid'].apply(lambda x: x[1])

In [ ]:
#Convert eaglei_outages to an xarray DataArray with LON and LAT and datetime as coordinates and using all its variables as data variables
eaglei_outages_da = xr.DataArray(
    eaglei_outages.drop(columns=['centroid']),
    coords={
        'longitude': ('points', eaglei_outages['LON'].values),
        'latitude': ('points', eaglei_outages['LAT'].values),
        'time': ('points', eaglei_outages['time'].values)
    },
    dims='points'
)

KeyError: 'time'